# Incremental

The incremental materialisation strategy assumes that at first run dbt materialises model in the database and then updates it accroding to a specific logic.

The logic is determined by the **incremental strategy**. This section considers the different incremental strategies, their configurations, and underlying processes.

## Append

The `append` strategy adds entities for the next `dbt run` command.

---

Consider the example that shows the concept of the `append` incremental strategy.

In [1]:
# init
dbt init append --profile knowledge -q
cd append

In [2]:
# file append/seeds/data.csv
id,value
1,"value"

**Note** that the `append` strategy is the default for the `incremental` materialisation, but it is specified explicitly in this example.

In [3]:
# file append/models/experimental.sql
{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'append',
    )
}}

select *
from {{ ref('data') }} as source_data

The first run of the model:

In [4]:
dbt seed -q --full-refresh
dbt run -q --select experimental --full-refresh
dbt show -q --inline "select * from {{ ref('experimental') }}"

| id | value |
| -- | ----- |
|  1 | value |



Each subsequent run simply appends the records retrieved from the query:

In [5]:
dbt run -q --select experimental
dbt show -q --inline "select * from {{ ref('experimental') }}"

| id | value |
| -- | ----- |
|  1 | value |
|  1 | value |



## Delete+insert

The `delete+insert` strategy replaces the existing data with the new query records. The columns defined in the unique key are used to identify the records that have to be replaced.

---

Consider the example that shows the main idea of the `delete+insert` strategy:

In [1]:
# init
dbt init delete_insert --profile knowledge -q
cd delete_insert

In [3]:
# file delete_insert/seeds/data.csv
id,value
1,"value1"
2,"value2"

The model that follows `delete+insert` incremental strategy:

In [4]:
# file delete_insert/models/experimental.sql
{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'delete+insert',
        unique_key = ['id']
    )
}}

select *
from {{ ref('data') }} as source_data

The `unieque_key = ['id']` column is used to identify new records.

The intial run of the model:

In [5]:
dbt seed -q --full-refresh
dbt run -q --full-refresh --select experimental
dbt show -q --inline "select * from {{ ref('experimental') }}"

| id | value  |
| -- | ------ |
|  1 | value1 |
|  2 | value2 |



Data update:

In [6]:
# file delete_insert/seeds/data.csv
id,value
2,"new value"
3,"value3"

In [7]:
dbt seed -q --full-refresh
dbt run -q --select experimental
dbt show -q --inline "select * from {{ ref('experimental') }}"

| id | value     |
| -- | --------- |
|  1 | value1    |
|  2 | new value |
|  3 | value3    |



Note that the value under `id=2` is updated according to the incremental run.